# NLP Task 5 — Fine-Tuning BERT for POS Tagging & Chunking
**Internship Assignment | Token Classification using Transformers**

## Task 1: Dataset Selection

In [ ]:
# Install required libraries
!pip install transformers datasets seqeval evaluate accelerate -q

In [ ]:
from datasets import load_dataset

# Load CoNLL-2003 dataset (Parquet version without script) — contains both POS tags and Chunk tags
dataset = load_dataset("eriktks/conll2003")
print(dataset)

In [ ]:
# Inspect the features to understand label types
print("Features:", dataset["train"].features)
print("\nSample entry:")
print(dataset["train"][0])

In [ ]:
# POS tag list from CoNLL-2003
pos_label_list = dataset["train"].features["pos_tags"].feature.names
print("POS Labels:", pos_label_list)

# Chunk tag list from CoNLL-2003
chunk_label_list = dataset["train"].features["chunk_tags"].feature.names
print("\nChunk Labels:", chunk_label_list)

## Task 2: Data Preprocessing

In [ ]:
from transformers import AutoTokenizer

# Use DistilBERT tokenizer (faster, lighter than full BERT)
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
# We'll train two separate models: one for POS tagging, one for Chunking
# This function aligns labels after BERT subword tokenization

def tokenize_and_align_labels(examples, task="pos"):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,  # input is already word-tokenized
        padding="max_length",
        max_length=128
    )

    all_labels = []
    tag_key = "pos_tags" if task == "pos" else "chunk_tags"

    for i, label in enumerate(examples[tag_key]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens [CLS] and [SEP] get -100 (ignored in loss)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word gets the real label
                label_ids.append(label[word_idx])
            else:
                # Subsequent subwords of the same word also get -100
                label_ids.append(-100)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [ ]:
# Tokenize and align for POS tagging
tokenized_pos = dataset.map(
    lambda x: tokenize_and_align_labels(x, task="pos"),
    batched=True
)

# Tokenize and align for Chunking
tokenized_chunk = dataset.map(
    lambda x: tokenize_and_align_labels(x, task="chunk"),
    batched=True
)

print("POS tokenized sample keys:", tokenized_pos["train"].features.keys())

In [ ]:
# Verify: Check input_ids, attention_mask, and labels for one sample
sample = tokenized_pos["train"][0]
print("input_ids[:10]   :", sample["input_ids"][:10])
print("attention_mask[:10]:", sample["attention_mask"][:10])
print("labels[:10]      :", sample["labels"][:10])

## Task 3: Model Setup

In [ ]:
from transformers import AutoModelForTokenClassification

# --- POS Tagging Model ---
num_pos_labels = len(pos_label_list)
pos_id2label = {i: label for i, label in enumerate(pos_label_list)}
pos_label2id = {label: i for i, label in enumerate(pos_label_list)}

pos_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_pos_labels,
    id2label=pos_id2label,
    label2id=pos_label2id
)

print(f"POS Model loaded | Labels: {num_pos_labels}")

In [ ]:
# --- Chunking Model ---
num_chunk_labels = len(chunk_label_list)
chunk_id2label = {i: label for i, label in enumerate(chunk_label_list)}
chunk_label2id = {label: i for i, label in enumerate(chunk_label_list)}

chunk_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_chunk_labels,
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

print(f"Chunk Model loaded | Labels: {num_chunk_labels}")

## Task 4: Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Data collator handles dynamic padding within each batch
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
# Load seqeval for token-level evaluation
seqeval = evaluate.load("seqeval")

def make_compute_metrics(label_list):
    """Factory function to create metrics function for a given label set."""
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        # Remove -100 (special tokens) and convert ids to label strings
        true_predictions = [
            [label_list[pred] for pred, lab in zip(preds, labs) if lab != -100]
            for preds, labs in zip(predictions, labels)
        ]
        true_labels = [
            [label_list[lab] for pred, lab in zip(preds, labs) if lab != -100]
            for preds, labs in zip(predictions, labels)
        ]

        results = seqeval.compute(predictions=true_predictions, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"]
        }
    return compute_metrics

In [ ]:
# ---- Training Arguments (shared for both models) ----
def get_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_dir="./logs",
        logging_steps=100,
        report_to="none"  # disable wandb
    )

In [ ]:
# ---- Train POS Tagging Model ----
pos_trainer = Trainer(
    model=pos_model,
    args=get_training_args("./pos_model"),
    train_dataset=tokenized_pos["train"],
    eval_dataset=tokenized_pos["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_label_list)
)

print("Starting POS Tagging Training...")
pos_trainer.train()

In [ ]:
# ---- Train Chunking Model ----
chunk_trainer = Trainer(
    model=chunk_model,
    args=get_training_args("./chunk_model"),
    train_dataset=tokenized_chunk["train"],
    eval_dataset=tokenized_chunk["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_label_list)
)

print("Starting Chunking Training...")
chunk_trainer.train()

## Task 5: Evaluation

In [ ]:
# Evaluate POS Tagging model on test split
print("=== POS Tagging Evaluation ===")
pos_results = pos_trainer.evaluate(eval_dataset=tokenized_pos["test"])
print(f"Precision : {pos_results['eval_precision']:.4f}")
print(f"Recall    : {pos_results['eval_recall']:.4f}")
print(f"F1 Score  : {pos_results['eval_f1']:.4f}")
print(f"Accuracy  : {pos_results['eval_accuracy']:.4f}")

In [ ]:
# Evaluate Chunking model on test split
print("=== Chunking Evaluation ===")
chunk_results = chunk_trainer.evaluate(eval_dataset=tokenized_chunk["test"])
print(f"Precision : {chunk_results['eval_precision']:.4f}")
print(f"Recall    : {chunk_results['eval_recall']:.4f}")
print(f"F1 Score  : {chunk_results['eval_f1']:.4f}")
print(f"Accuracy  : {chunk_results['eval_accuracy']:.4f}")

## Task 6: Inference

In [ ]:
import torch

def predict_tags(sentence, model, tokenizer, label_list):
    """Run inference and map predicted label ids back to label strings."""
    words = sentence.split()
    
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    
    predicted_ids = torch.argmax(outputs.logits, dim=2)[0].tolist()
    word_ids = inputs.word_ids(batch_index=0)

    # Collect only one prediction per original word
    result = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in result:
            result[word_id] = label_list[predicted_ids[idx]]

    return list(result.values())

In [ ]:
# Test sentence
sentence = "John works at Google in California"
words = sentence.split()

pos_tags = predict_tags(sentence, pos_model, tokenizer, pos_label_list)
chunk_tags = predict_tags(sentence, chunk_model, tokenizer, chunk_label_list)

print("Input Sentence :", sentence)
print()
print(f"{'Word':<15} {'POS Tag':<15} {'Chunk Tag'}")
print("-" * 45)
for word, pos, chunk in zip(words, pos_tags, chunk_tags):
    print(f"{word:<15} {pos:<15} {chunk}")

In [ ]:
# Test with more custom sentences
test_sentences = [
    "The quick brown fox jumps over the lazy dog",
    "Apple released a new iPhone in September",
    "She is studying machine learning at university"
]

for sent in test_sentences:
    words = sent.split()
    pos_preds = predict_tags(sent, pos_model, tokenizer, pos_label_list)
    chunk_preds = predict_tags(sent, chunk_model, tokenizer, chunk_label_list)
    
    print(f"\nSentence: {sent}")
    print(f"{'Word':<20} {'POS':<15} {'Chunk'}")
    print("-" * 50)
    for w, p, c in zip(words, pos_preds, chunk_preds):
        print(f"{w:<20} {p:<15} {c}")

## Task 7: Comparison — POS Tagging vs Chunking

In [ ]:
import pandas as pd

# Side-by-side comparison of what each task does
comparison_data = {
    "Aspect": [
        "Task Type",
        "Granularity",
        "Output",
        "Example Output",
        "Difficulty",
        "Label Count (CoNLL-2003)",
        "Eval F1 Score"
    ],
    "POS Tagging": [
        "Grammar-level labeling",
        "Word-level",
        "Tag per word (NN, VBZ, NNP...)",
        "John=NNP, works=VBZ",
        "Easier",
        str(len(pos_label_list)),
        f"{pos_results['eval_f1']:.4f}"
    ],
    "Chunking": [
        "Phrase-level grouping",
        "Span-level (BIO scheme)",
        "Phrase boundaries (B-NP, I-NP, B-VP...)",
        "John=B-NP, works=B-VP",
        "Moderate",
        str(len(chunk_label_list)),
        f"{chunk_results['eval_f1']:.4f}"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visual comparison of evaluation metrics
metrics = ["Precision", "Recall", "F1 Score"]
pos_scores = [
    pos_results["eval_precision"],
    pos_results["eval_recall"],
    pos_results["eval_f1"]
]
chunk_scores = [
    chunk_results["eval_precision"],
    chunk_results["eval_recall"],
    chunk_results["eval_f1"]
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, pos_scores, width, label="POS Tagging", color="steelblue")
bars2 = ax.bar(x + width/2, chunk_scores, width, label="Chunking", color="tomato")

ax.set_ylabel("Score")
ax.set_title("POS Tagging vs Chunking — Evaluation Metrics")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
ax.bar_label(bars1, fmt="%.3f", padding=3)
ax.bar_label(bars2, fmt="%.3f", padding=3)

plt.tight_layout()
plt.savefig("pos_vs_chunk_comparison.png", dpi=150)
plt.show()

## Task 8: Report / Blog

### Differences Between POS Tagging and Chunking

**Part-of-Speech (POS) Tagging** assigns a grammatical category (noun, verb, adjective, etc.) to each individual word in a sentence. It works at the word level. For example, in "John works at Google", POS tagging tells us that *John* is a proper noun (NNP) and *works* is a verb (VBZ).

**Chunking** (also called shallow parsing) groups words into meaningful phrases like noun phrases (NP) or verb phrases (VP). It uses the BIO tagging scheme — B-NP means the beginning of a noun phrase, I-NP means inside it. Chunking works at the span level and depends on POS tags conceptually.

### Challenges Faced

1. **Subword tokenization alignment** — BERT breaks words into subword tokens (e.g., "running" → "run", "##ning"). Labels must only be assigned to the first subword; subsequent subwords get -100 to be ignored by the loss function.

2. **BIO scheme consistency in chunking** — Chunk tags follow a Begin-Inside-Outside scheme. Misaligned predictions can break phrase boundaries, making evaluation sensitive.

3. **Class imbalance** — Some POS tags (like punctuation or conjunctions) appear far less than nouns and verbs, making it harder to generalize.

### Observations and Insights

- DistilBERT trains significantly faster than full BERT while achieving competitive performance — a great trade-off for internship-scale experiments.
- POS tagging converges faster and reaches higher F1 scores because each word has one unambiguous label in most cases.
- Chunking is slightly harder since it requires understanding context across multiple words to define phrase boundaries.
- The seqeval metric is much stricter than token-level accuracy — it requires the entire sequence of labels in a span to be correct before counting it as a true positive, which is a more realistic real-world measure.
- Pre-trained transformer models require minimal training data to achieve strong results on structured NLP tasks like these, validating the power of transfer learning.

In [ ]:
# Save both models for future use
pos_model.save_pretrained("./saved_pos_model")
tokenizer.save_pretrained("./saved_pos_model")
print("POS model saved.")

chunk_model.save_pretrained("./saved_chunk_model")
tokenizer.save_pretrained("./saved_chunk_model")
print("Chunk model saved.")

In [ ]:
# Final Summary
print("="*55)
print("          FINAL RESULTS SUMMARY")
print("="*55)
print(f"{'Metric':<20} {'POS Tagging':>15} {'Chunking':>15}")
print("-"*55)
print(f"{'Precision':<20} {pos_results['eval_precision']:>15.4f} {chunk_results['eval_precision']:>15.4f}")
print(f"{'Recall':<20} {pos_results['eval_recall']:>15.4f} {chunk_results['eval_recall']:>15.4f}")
print(f"{'F1 Score':<20} {pos_results['eval_f1']:>15.4f} {chunk_results['eval_f1']:>15.4f}")
print(f"{'Accuracy':<20} {pos_results['eval_accuracy']:>15.4f} {chunk_results['eval_accuracy']:>15.4f}")
print("="*55)